# DuckDB - OneLake Files

In this demo we query files in OneLake using DuckDB.

The use case being:

You are a data scientist who wants to explore the feasibility of developing a machine learning model over data that is available in OneLake.
You want to do that exploration on your local laptop so you can use Jupyter notebook in VS Code and get a slick experience when you run ML models locally over the data.

Pre-requisites: you have uploaded some data onto OneLake in CSV format that you want to query.

In [12]:
import duckdb

import os

from azure.storage.filedatalake import (
    DataLakeServiceClient,
    DataLakeDirectoryClient,
    FileSystemClient
)
from azure.identity import DefaultAzureCredential


In [13]:
ACCOUNT_NAME = "onelake"
WORKSPACE_NAME = "DuckDbPolarsDemos"
LAKEHOUSE_NAME = "PricePaidData"

# Files
FOLDER = "land_registry"
SINGLE_FILE = "pp-2023.csv"
ALL_FILES = "pp-*.csv"

In [14]:
abfss_path = f"abfss://{WORKSPACE_NAME}@{ACCOUNT_NAME}.dfs.fabric.microsoft.com/{LAKEHOUSE_NAME}.Lakehouse/Files/{FOLDER}/{SINGLE_FILE}"

In [15]:
# Tables
SCHEMA = "land_registry"
TABLE_DUCKDB_PARQUET = "house_sales_duckdb_parquet/data.parquet"
TABLE_POLARS_DELTA = "house_sales_polars_delta"

In [16]:
account_url = f"https://{ACCOUNT_NAME}.dfs.fabric.microsoft.com"
abfss_path = f"abfss://{WORKSPACE_NAME}@{ACCOUNT_NAME}.dfs.fabric.microsoft.com/{LAKEHOUSE_NAME}.Lakehouse/Files/{FOLDER}/{SINGLE_FILE}"

## Conventional connection to Onelake

Show how we can connect to and download data from OneLake using conventional Pyton SDK.

In [17]:
token_credential = DefaultAzureCredential()
service_client = DataLakeServiceClient(account_url, credential=token_credential)

#Create a file system client for the workspace
file_system_client = service_client.get_file_system_client(WORKSPACE_NAME)

In [18]:
#List a directory within the filesystem
paths = file_system_client.get_paths(path=f"{LAKEHOUSE_NAME}.Lakehouse/Files/{FOLDER}")

for path in paths:
    print(path.name)

PricePaidData.Lakehouse/Files/land_registry/pp-2000.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2001.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2002.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2003.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2004.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2005.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2006.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2007.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2008.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2009.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2010.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2011.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2012.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2013.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2014.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2015.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2016.csv
PricePaidData.Lakehouse/Files/land_registry/pp-2

In [19]:
# Download only the first 5 lines from the remote file in OneLake
file_client = file_system_client.get_file_client(f"{LAKEHOUSE_NAME}.Lakehouse/Files/{FOLDER}/{SINGLE_FILE}")

# Read the first 4096 bytes (adjust if needed for very long lines)
chunk_size = 4096
partial_download = file_client.download_file(offset=0, length=chunk_size)
chunk = partial_download.readall().decode('utf-8')
lines = chunk.splitlines()[:5]
for line in lines:
    print(line)

"{06C9F487-D94B-9388-E063-4804A8C0BD98}","330000","2023-03-20 00:00","CF14 7BX","T","N","F","32","","HEOL PANT Y CELYN","","CARDIFF","CARDIFF","CARDIFF","A","A"
"{06C9F487-D94C-9388-E063-4804A8C0BD98}","269950","2023-07-25 00:00","LL28 4SH","D","N","F","7","","MARSTON DRIVE","RHOS ON SEA","COLWYN BAY","CONWY","CONWY","A","A"
"{06C9F487-D94D-9388-E063-4804A8C0BD98}","280000","2023-08-10 00:00","LL31 9BN","D","N","F","PLAS COLWYN","","LLYS HELYG","DEGANWY","CONWY","CONWY","CONWY","A","A"
"{06C9F487-D94E-9388-E063-4804A8C0BD98}","699999","2023-08-24 00:00","SA62 6BA","D","N","F","MIDDLE LOCHVANE","","","PEN Y CWM","HAVERFORDWEST","PEMBROKESHIRE","PEMBROKESHIRE","A","A"
"{06C9F487-D94F-9388-E063-4804A8C0BD98}","160000","2023-08-21 00:00","SY16 1QY","T","N","F","167","","LON DOLAFON","","NEWTOWN","POWYS","POWYS","A","A"


## DuckDB connection to OneLake

Insprired by LinkedIn post:

https://www.linkedin.com/posts/mimounedjouallah_onelake-duckdb-powerquery-activity-7319649180493180928-1Jbu/

Pasted below for convenience:

🚀 Want to query data from hashtag#OneLake using hashtag#DuckDB on your laptop?

Here’s all you need to do (just once!):

1️⃣ Install Azure CLI
2️⃣ Run az login
3️⃣ Execute this SQL in DuckDB: ( the key word is PERSISTENT )

`CREATE or replace PERSISTENT SECRET onelake (TYPE azure,PROVIDER credential_chain, CHAIN 'cli',ACCOUNT_NAME 'onelake');`

That’s it! 🎉 From now on, every time you use DuckDB (CLI, Python, or the hashtag#Powerquery connector)

it will automatically fetch the token to access your data.

📌 In hashtag#MicrosoftFabric hashtag#Python notebooks, this is handled for you automatically.

the implication is, the same SQL script or python code will run exactly the same, either your run it locally or in the service !!!

## Set up DuckDB

In [20]:
db = duckdb.connect(database=":memory:", read_only=False)

## Set up secret in DuckDB to enable connection to OneLake.


In [21]:
db.sql("CREATE OR REPLACE PERSISTENT SECRET onelake (TYPE azure, PROVIDER credential_chain, CHAIN 'cli', ACCOUNT_NAME 'onelake');")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [22]:
db.sql("FROM duckdb_secrets()")

┌─────────┬─────────┬──────────────────┬────────────┬────────────┬──────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│  name   │  type   │     provider     │ persistent │  storage   │                scope                 │                                                              secret_string                                                               │
│ varchar │ varchar │     varchar      │  boolean   │  varchar   │              varchar[]               │                                                                 varchar                                                                  │
├─────────┼─────────┼──────────────────┼────────────┼────────────┼──────────────────────────────────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ onelake │ azure   

In [23]:
# Key step when running in a Linux environment
db.sql("SET azure_transport_option_type = 'curl'")

## Run query directly over OneLake


In [24]:
db.sql(
    f"""
    SELECT
    column00 AS 'id',
    column01 AS 'price',
    column02 AS 'date',
    column03 AS 'postcode',
    column04 AS 'property_type',
    column05 AS 'old_new',
    column06 AS 'duration',
    column07 AS 'paon',
    column08 AS 'saon',
    column09 AS 'street',
    column10 AS 'locale',
    column11 AS 'town_city',
    column12 AS 'district',
    column13 AS 'county',
    column14 AS 'ppd_category',
    column15 AS 'record_type',
    year(date) AS 'year_of_sale',
    month(date) AS 'month_of_sale'
    FROM read_csv(
        '{abfss_path}',
        header=False
    );
    """
)

┌────────────────────────────────────────┬────────┬─────────────────────┬──────────┬───────────────┬─────────┬──────────┬─────────────────┬─────────────┬────────────────────────┬────────────────┬──────────────────┬───────────────┬───────────────┬──────────────┬─────────────┬──────────────┬───────────────┐
│                   id                   │ price  │        date         │ postcode │ property_type │ old_new │ duration │      paon       │    saon     │         street         │     locale     │    town_city     │   district    │    county     │ ppd_category │ record_type │ year_of_sale │ month_of_sale │
│                varchar                 │ int64  │      timestamp      │ varchar  │    varchar    │ varchar │ varchar  │     varchar     │   varchar   │        varchar         │    varchar     │     varchar      │    varchar    │    varchar    │   varchar    │   varchar   │    int64     │     int64     │
├────────────────────────────────────────┼────────┼─────────────────────┼──────